In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys
sys.path.append('..')
import pandas as pd
import src.observables as obs
import utils.io as io

In [2]:
DATA_ROOT = '../../data/iDMRG/'
REQ_DATA = 'scanChi800'
PATH_TO_REQ_DATA = os.path.join(DATA_ROOT, REQ_DATA)
file_list = [os.path.join(PATH_TO_REQ_DATA,f) for f in os.listdir(PATH_TO_REQ_DATA) if ".h5" in f][0:1]

def atomic_action(file, distrange=None, storepsi=False): 
    psi, metadata = io.load_mps_with_metadata(file)
    model_params = metadata['model_params']
    correlation_length = psi.correlation_length2()
    if not distrange:
     distrange=int(0.9 * correlation_length)
    charge_corr = obs.charge_corr(psi, distrange, connected=True)
    spinz_corr = obs.SzSz_corr(psi, distrange, connected=True)
    spspm_corr = obs.SpSm_corr(psi, distrange)
    electron_corr = obs.electron_corr(psi, distrange)
    _, spin2_corr = obs.op_correlation_idmrg(psi, xs=np.arange(1, distrange))
    row_dict = { 
        'dV' : model_params['Vpm'] - model_params['Vpp'], 
        'mean_Sz' : np.mean(psi.expectation_value("Sz")), 
        'mean_fill' : np.mean(psi.expectation_value("Ntot")),
        'model_params' : model_params, 
        'diagnostics' : metadata['diagnostics'], 
        'xi' : correlation_length,
        'charge_corr' : charge_corr, 
        'spinz_corr' : spinz_corr, 
        'spsm_corr' : spspm_corr, 
        'electron_corr' : electron_corr, 
        'spin2_corr'  : spin2_corr,
    }
    if storepsi: 
        row_dict['psi'] = psi
    return row_dict
    
df_i = pd.DataFrame([atomic_action(f) for f in file_list])
df_i = df_i.sort_values('dV')

In [7]:
seldf = df_i.iloc[0]
len(seldf['charge_corr']), len(seldf['spin2_corr'])

(180, 179)